[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/thermocouple/thermocouple.ipynb)

# Thermocouple

This example computes coupled charge and heat transport through a two-metal thermocouple bent into a planar hairpin. Heating the junction between two dissimilar metals produces an open-circuit voltage across the free ends, which is called the Seebeck effect, while driving an electric current through the junction cools or warms it, which is called the Peltier effect. Conduction electrons carry both electric charge and entropy, whereas lattice vibrations carry entropy alone, and the electrons in the two metals carry different entropies per unit charge. When the junction is heated with open ends, each leg builds up the electric field needed to hold its electrons against the temperature fall, leaving the difference between the two legs as a voltage across the feet. When a current passes through the junction instead, electrons arrive through one metal carrying one entropy and leave through the other carrying a different entropy, cooling or warming the junction according to the current direction.

Planar projective spacetime is extended by a metric-free direction for entropy, so the electrochemical potential along time and the temperature along entropy combine into one potential 1-form whose spatial fall `-gradients ^ Potential` is a single driving bivector. The term ***parcel*** here means the extensive bundle of charge and entropy carried together by one carrier. Joining a carrier velocity with its parcel defines the advection map `Velocity.regressive(parcels)` from velocities to fluxes, and its adjugate maps a drive to the mechanical force on the carrier so `drive & advection(velocity) == advection.adjugate()(drive) & velocity`. Composing advection with an isotropic mobility gives a self-adjugate local response `advection(mobility(advection.adjugate()))`, and pulling that response back across the triangle slopes yields a self-adjugate vertex conductance `gradient.adjugate()((areas * responses)(gradient))` that equates Seebeck voltage per unit heating to Peltier cooling per unit current at every instant.

In linear irreversible thermodynamics, charge and heat transport split into two coupled vector constitutive equations, $\mathbf{J} = \sigma(\mathbf{E} - S\nabla T)$ and $\mathbf{J}_Q = \Pi\mathbf{J} - \kappa\nabla T$, where the Seebeck coefficient $S$ and Peltier coefficient $\Pi$ are linked separately by Kelvin's reciprocal relation $\Pi = T S$. Solving the transient profile across the two-metal domain then couples current continuity $\nabla\cdot\mathbf{J} = 0$ to an entropy balance $c\,\partial_t T + \nabla\cdot(\mathbf{J}_Q / T) = \dot s$ across the junction.

In [ ]:
# Install numga and its examples when running on Google Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag
from examples.util.animation import show_animation
from examples.thermodynamics.thermocouple import render, scenarios

# Planar projective spacetime joined with a metric-free direction for entropy.
ga = Algebra("x+y+t-w0") * Algebra("s0")
space = Algebra("x+y+")
quantities = Algebra("t-s0")
mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()                                                    # Vector+
Parcel = Plane.derive.within(quantities).derive.dual                          # Antivector-
Velocity = Plane.derive.within(space).derive.dual                             # Antivector-
Force = Plane.derive.within(space)                                            # Vector+
Potential = Plane.derive.within(quantities)                                   # Vector+
time = -mv.t.dual()                                                           # [] Parcel-
entropy = mv.s.dual()                                                         # [] Parcel-
np.set_printoptions(precision=3, suppress=True)

## Material transport response

Each metal carries conduction electrons with charge and entropy alongside neutral lattice vibrations with entropy alone. Multiplying a carrier velocity by `Velocity` turns the regressive join with its parcel into a linear advection map from velocities to fluxes, while its adjugate reverses the map under the regressive pairing `drive & advection(velocity) == advection.adjugate()(drive) & velocity` to give the force exerted on the carrier by a drive. An isotropic carrier mobility `Force.dual()` maps that force to a drift velocity, and summing the composed map over both carrier species yields each metal's transport response.

In [ ]:
# Carrier charges, per-metal entropies, and mobilities for conduction electrons and lattice vibrations.
charges = np.array([-1, 0])                                                   # [2]
entropies = np.array([[2.0, 1.0], [0.5, 1.0]])                                # [2, 2]
mobilities = np.array([1.0, 1.0])                                             # [2]

In [ ]:
parcels = charges * time + entropies * entropy                                # [2, 2] Parcel-
# Joining a carrier velocity with its parcel forms its flux, while the adjugate maps a drive to the carrier force.
advection = Velocity.regressive(parcels)                                      # [2, 2] Antibivector- <- Antivector-
mobility = mv.scalar(mobilities[:, None]) * Force.dual()                      # [2] Antivector- <- Vector+
responses = advection(mobility(advection.adjugate())).sum(axis=-1)            # [2] Antibivector- <- Bivector+
# Pair unit electrochemical-potential and temperature falls along x against the resulting fluxes in the first metal.
drives = -mv.x ^ stack((mv.t, mv.s))                                          # [2] Bivector+
transport_rates = drives[:, None] & responses[0](drives)                      # [2, 2] Scalar+

print("charge and entropy driven by electrochemical potential fall:", transport_rates.to_array()[:, 0])
print("charge and entropy driven by temperature fall:              ", transport_rates.to_array()[:, 1])

A unit fall of the electrochemical potential drives one unit of charge and minus two units of entropy because negatively charged electrons drift up the potential carrying their entropy along. A unit fall of temperature drives the same minus two units of charge together with five units of entropy, four carried by electrons and one by lattice vibrations. Both off-diagonal readings agree because the material response equals its own adjugate.

## Hairpin geometry and simplex gradients

Two legs of dissimilar metals meet across the top bridge of a triangulated planar hairpin whose two feet rest on a cold reference block while heat enters along the top of the junction. The term ***simplex gradient*** here means the spatial slope on a triangle of the affine function equal to one at one corner and zero on the opposite side. Joining the opposite side with the time and entropy directions leaves a hyperplane invariant along the non-spatial axes, and dividing by the full join with the corner yields that corner's simplex gradient.

In [ ]:
topology, vertices, metals, first_foot, second_foot, junction_vertices = scenarios.hairpin(scenarios.spacing)
V, E, F = topology.n_elements
corners = topology.incidence[2, 0] * vertices                                 # [3] Antivector-[F]
facing = topology.off[2, 0] * vertices                                        # [2, 3] Antivector-[F]
sides = facing[0] & facing[1]                                                 # [3] Antibivector-[F]
# Joining each opposite side with time and entropy forms a hyperplane whose quotient by the corner join equals one
# at that corner and zero on the opposite side; casting to Force drops the origin offset along w.
gradients = ((sides & time & entropy) / (sides & time & entropy & corners)).cast(Force)   # [3] Force+[F]
areas = (corners[0] & corners[1] & corners[2] & time & entropy).norm() / 2    # Scalar+[F]
render.hairpin_figure(topology, vertices, metals, (first_foot, second_foot), scenarios.resolution);

The left leg belongs to the first metal and the right leg to the second, with the top line marking the bimetallic junction and the gold strokes marking the two contact feet on the cold block.

## Vertex conductance and entropy storage

Wedging each corner's simplex gradient with the vertex potential maps vertex potentials to driving bivectors on every triangle, and its adjugate collects triangle fluxes into net parcel outflow from each vertex's dual cell. Composing those two maps around the area-weighted triangle responses yields the hairpin's vertex conductance. Each vertex's dual cell also stores entropy proportional to its area and temperature rise without storing net charge, while strong thermal clamping on the second foot drains entropy to the cold block without pinning its electric potential.

In [ ]:
capacity, clamping = scenarios.capacity, scenarios.clamping

In [ ]:
corner_incidence = topology.incidence[2, 0]                                   # [3] Scalar+[F <- V]
gradient = (spdiag(gradients ^ Potential) * corner_incidence).sum(axis=0)     # (Bivector+ <- Vector+)[F <- V]
conductance_map = gradient.adjugate()((areas * responses[metals].field())(gradient))   # (Parcel- <- Potential+)[V <- V]
# Pairing a potential with unit entropy reads the temperature rise and stores proportional entropy without charge.
unit_capacity = entropy * (Potential & entropy)                               # [] Parcel- <- Potential+
dual_areas = (corner_incidence.adjugate() * (areas / 3)).sum(axis=0)          # Scalar+[V]
clamped_foot = mv.scalar(np.isin(np.arange(V), second_foot)[:, None]).field()   # Scalar+[V]
storage_map = spdiag(dual_areas * capacity * unit_capacity)                   # (Parcel- <- Potential+)[V <- V]
loss_map = conductance_map + spdiag(dual_areas * clamped_foot * clamping * unit_capacity)   # (Parcel- <- Potential+)[V <- V]

## Seebeck heating and Peltier current

Two separate implicit backward-Euler runs step the hairpin from equilibrium with the first foot pinned to zero potential. One run injects entropy at a constant rate along the top of the junction, while the other draws a constant electric current out of the second foot within linear response around equilibrium. Stepping and rendering the seventy-five frames takes about three seconds.

In [ ]:
heating, current = scenarios.heating, scenarios.current
n_frames, frame_duration_ms, time_step = scenarios.n_frames, scenarios.frame_duration_ms, scenarios.time_step

In [ ]:
def settling_potentials(sources: Parcel) -> Iterator[Potential]:
    """Yields vertex potentials (`Potential+[V]`) across `n_frames` backward-Euler steps with `first_foot` pinned to zero."""
    free = topology.reading(np.setdiff1d(np.arange(V), first_foot), 0)        # Scalar+[V_free <- V]
    step_map = free * (loss_map + storage_map / time_step) * free.adjugate()   # (Parcel- <- Potential+)[V_free <- V_free]
    # Widen sources and initial potentials to their full charge-and-entropy layouts before stepping.
    zero_field = 0 * ((mv.t + mv.s) & sources)                                # Scalar+[V]
    sources = sources + zero_field * (time + entropy)                         # Parcel-[V]
    potentials = zero_field * (mv.t + mv.s)                                   # Potential+[V]
    for _ in range(n_frames):
        yield potentials
        potentials = free.adjugate() * step_map.solve(free * (sources + storage_map(potentials) / time_step))


shares = np.zeros((2, V))
shares[0, junction_vertices], shares[1, second_foot] = 1 / len(junction_vertices), 1 / len(second_foot)
junction_weights, foot_weights = mv.scalar(shares[..., None]).field()         # Scalar+[V], Scalar+[V]
frames = list(
    zip(
        settling_potentials(junction_weights * heating * entropy),
        settling_potentials(foot_weights * -current * time),
    )
)
show_animation(
    render.animation(topology, vertices, metals, (first_foot, second_foot), scenarios.resolution, frames[-1], frames),
    "thermocouple",
    frame_duration_ms,
    n_frames=n_frames,
)

Junction heating appears on the left and current-driven cooling on the right, each tinted from blue below equilibrium through grey to warm red with level contours of the electrochemical potential overlaid. Under junction heating, warmth spreads down both legs with zero net current, so each leg builds the potential slope that balances thermal drift for its own electrons, crowding level contours in the left leg relative to the right and leaving a net voltage across the feet. Under current drive, electrons enter the junction from the right leg carrying little entropy and depart down the left leg carrying four times as much, cooling the junction while lattice conduction draws warmth up both legs from the block.

## Voltage and cooling over time

Pairing the recorded states with unit charge on the second foot reads the open-circuit Seebeck voltage per unit heating, while pairing with unit entropy along the top of the junction reads the Peltier temperature drop per unit current.

In [ ]:
heated_states = stack([potentials for potentials, _ in frames])               # [n_frames] Potential+[V]
driven_states = stack([potentials for _, potentials in frames])               # [n_frames] Potential+[V]
voltage = (topology.reading(second_foot, 0) * (heated_states & time)).batch().mean(axis=-1) / heating             # [n_frames] Scalar+
cooling = -(topology.reading(junction_vertices, 0) * (driven_states & entropy)).batch().mean(axis=-1) / current   # [n_frames] Scalar+
render.readings_figure(time_step * np.arange(n_frames), voltage, cooling);

The solid curve shows the heated hairpin's foot voltage per unit heating and the markers show the current-driven hairpin's junction cooling per unit current. Both curves coincide throughout the transient relaxation as well as in steady state because the vertex loss and entropy-storage operators are each equal to their own adjugate.

In [ ]:
# checks
velocity = mv.x.dual()                                                        # [] Velocity-
force = advection.adjugate()(drives[1])                                       # [2, 2] Force+
assert (((drives[1] & advection(velocity)) - (force & velocity)).abs() < 1e-12).all()
np.testing.assert_allclose(responses.adjugate().kernel, responses.kernel, atol=1e-12)
assert ((drives & responses[0](drives)) > 0).all()
# Every unit parcel drifts down its own potential gradient along -x.
units = stack((time, entropy))                                                # [2] Parcel-
assert ((mv.x & Force.dual()((Velocity & units).adjugate()(drives))) < 0).all()
# Vertex loss is self-adjugate under the vertex pairing, tying transient Seebeck voltage to Peltier cooling.
heated_state, driven_state = frames[-1]
heated_pairing = (heated_state & loss_map(driven_state)).sites.sum()
driven_pairing = (driven_state & loss_map(heated_state)).sites.sum()
assert (heated_pairing - driven_pairing).abs() < 1e-9 * heated_pairing.abs()
assert ((voltage - cooling).abs() < 1e-9).all()
assert voltage[-1] > 0
assert (voltage[-1] - voltage[-5]).abs() < 2e-2 * voltage[-1]